# Geospatial Embeddings for London Urban Prediction

This notebook builds the London PTAL and EPC downstream tasks, evaluates location and satellite embeddings, and continues with aerial image embeddings, controls, and residual analysis.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip -q install geopandas pyogrio rtree

In [ ]:
from pathlib import Path
import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd

from IPython.display import display

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    f1_score
)

pd.set_option("display.max_columns", 100)

## 1. Paths and run settings

The raw files are expected to be stored in Google Drive under:

`/content/drive/MyDrive/GEOG0105/Raw Data/`

The notebook saves only the main outputs that will be reused later. Preview tables and charts are only displayed inside the notebook.

In [ ]:
BASE = Path("/content/drive/MyDrive/GEOG0105")
RAW = BASE / "Raw Data"
OUT = BASE / "Outputs"

TABLE_DIR = OUT / "tables"
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY_DIR = RAW / "London_boundary"
POSTCODE_DIR = RAW / "Postcode_CSV"
PTAL_DIR = RAW / "PTAL"
EPC_FILE = RAW / "EPC Data.csv"

# Set this to False if the raw sample tables have already been built
# and you only want to rerun cleaning, baselines, and sampling.
REBUILD_SAMPLE_TABLES = True

# Coordinate baseline settings
RANDOM_STATE = 42
N_SPLITS = 5

# DINO/aerial extraction planning settings
PTAL_DINO_GRID_M = 500
EPC_DINO_TARGET_N = 20000

paths = {
    "BASE": BASE,
    "RAW": RAW,
    "BOUNDARY_DIR": BOUNDARY_DIR,
    "POSTCODE_DIR": POSTCODE_DIR,
    "PTAL_DIR": PTAL_DIR,
    "EPC_FILE": EPC_FILE
}

for name, path in paths.items():
    print(f"{name}: {path} | exists = {path.exists()}")

## 2. Helper functions

These functions standardise column names, postcodes, label classes, and repeated evaluation steps.

In [ ]:
def clean_postcode(pc):
    # Standardise UK postcodes for joining.
    if pd.isna(pc):
        return np.nan
    return str(pc).upper().replace(" ", "").strip()


def normalise_name(name):
    # Normalise column names for flexible matching.
    return "".join(ch for ch in str(name).lower() if ch.isalnum())


def pick_column(columns, candidates, required=True):
    # Pick a column using flexible case-insensitive matching.
    # It also handles hyphens and underscores.
    normalised = {normalise_name(c): c for c in columns}
    for candidate in candidates:
        key = normalise_name(candidate)
        if key in normalised:
            return normalised[key]
    if required:
        raise ValueError(
            "Could not find any of these columns: "
            f"{candidates}\nAvailable columns: {list(columns)}"
        )
    return None


def find_first_file(folder, pattern):
    files = list(Path(folder).rglob(pattern))
    if not files:
        raise FileNotFoundError(f"No file found for pattern {pattern} in {folder}")
    return files[0]


def mode_or_nan(series):
    series = series.dropna()
    if len(series) == 0:
        return np.nan
    return series.mode().iloc[0]


def ptal_to_level(value):
    # Group PTAL bands into low / medium / high accessibility.
    if pd.isna(value):
        return np.nan
    v = str(value).strip().lower()
    if v in ["0", "1a", "1b", "2"]:
        return "low"
    if v in ["3", "4"]:
        return "medium"
    if v in ["5", "6a", "6b"]:
        return "high"
    return np.nan


def epc_to_level(value):
    # Group EPC rating letters into low / medium / high energy performance.
    if pd.isna(value):
        return np.nan
    v = str(value).strip().upper()
    if v in ["A", "B"]:
        return "high"
    if v in ["C", "D"]:
        return "medium"
    if v in ["E", "F", "G"]:
        return "low"
    return np.nan


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def display_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)

## 3. Build the raw PTAL and EPC sample tables

This section reads the raw boundary, PTAL, postcode, and EPC files.  
The PTAL samples use the centroid of each 100m grid cell.  
The EPC samples are aggregated to postcode level because the available spatial join uses postcode centroids rather than exact property coordinates.

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Loading London borough boundaries")

    borough_candidates = list(BOUNDARY_DIR.rglob("*Borough*Excluding*MHW*.shp"))
    if len(borough_candidates) == 0:
        borough_candidates = list(BOUNDARY_DIR.rglob("*Borough*.shp"))

    print("Borough shapefile candidates:")
    for f in borough_candidates:
        print(" -", f)

    borough_shp = borough_candidates[0]
    boroughs_raw = gpd.read_file(borough_shp)
    boroughs_raw = boroughs_raw.to_crs(epsg=27700)

    print("\nUsing:", borough_shp)
    print("CRS:", boroughs_raw.crs)
    print("Shape:", boroughs_raw.shape)
    print("Columns:", boroughs_raw.columns.tolist())

    borough_name_col = pick_column(
        boroughs_raw.columns,
        ["NAME", "BOROUGH", "LAD_NAME", "DISTRICT", "NAME_1"],
        required=False
    )

    if borough_name_col is None:
        raise ValueError("Could not identify the borough name column. Please inspect the columns above.")

    boroughs = boroughs_raw[[borough_name_col, "geometry"]].rename(columns={borough_name_col: "borough"})
    boroughs["borough"] = boroughs["borough"].astype(str).str.strip()

    display(boroughs.head())
    print("Number of borough records:", len(boroughs))
else:
    print("Skipping raw rebuild. Existing sample tables will be loaded later.")

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Loading PTAL grid")

    ptal_shp = find_first_file(PTAL_DIR, "*.shp")
    print("Using PTAL shapefile:", ptal_shp)

    ptal_raw = gpd.read_file(ptal_shp)
    ptal_raw = ptal_raw.to_crs(epsg=27700)

    print("PTAL raw shape:", ptal_raw.shape)
    print("PTAL columns:", ptal_raw.columns.tolist())

    # Use centroid of each PTAL grid cell as the sample point.
    ptal_points = ptal_raw.copy()
    ptal_points["geometry"] = ptal_points.geometry.centroid

    access_col = pick_column(
        ptal_points.columns,
        ["AI", "Access Index", "ACCESS_INDEX", "AccessInde", "ACC_INDEX", "ACCESSINDE"],
        required=False
    )

    ptal_band_col = pick_column(
        ptal_points.columns,
        ["PTAL_2023", "PTAL", "PTAL2023", "PTAL_BAND", "PTALVALUE"],
        required=False
    )

    print("Access Index column:", access_col)
    print("PTAL band column:", ptal_band_col)

    ptal_joined = gpd.sjoin(
        ptal_points,
        boroughs,
        how="left",
        predicate="within"
    )

    ptal_joined["x"] = ptal_joined.geometry.x
    ptal_joined["y"] = ptal_joined.geometry.y

    ptal_wgs84 = ptal_joined.to_crs(epsg=4326)
    ptal_joined["lon"] = ptal_wgs84.geometry.x
    ptal_joined["lat"] = ptal_wgs84.geometry.y

    ptal_samples = pd.DataFrame({
        "sample_id": ["PTAL_" + str(i).zfill(7) for i in range(len(ptal_joined))],
        "task": "PTAL",
        "x": ptal_joined["x"].values,
        "y": ptal_joined["y"].values,
        "lon": ptal_joined["lon"].values,
        "lat": ptal_joined["lat"].values,
        "borough": ptal_joined["borough"].values,
        "crop_size_m": 300
    })

    if access_col is not None:
        ptal_samples["label_regression"] = pd.to_numeric(ptal_joined[access_col], errors="coerce")
    else:
        ptal_samples["label_regression"] = np.nan

    if ptal_band_col is not None:
        ptal_samples["label_classification"] = ptal_joined[ptal_band_col].astype(str)
    else:
        ptal_samples["label_classification"] = np.nan

    ptal_samples["ptal_level"] = ptal_samples["label_classification"].apply(ptal_to_level)

    print("PTAL samples:", ptal_samples.shape)
    print("PTAL samples without borough:", ptal_samples["borough"].isna().sum())
    display(ptal_samples.head())
else:
    print("Skipping PTAL raw rebuild.")

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Loading Code-Point Open postcode coordinates")

    postcode_files = sorted(list(POSTCODE_DIR.glob("*.csv")))
    print("Number of postcode CSV files:", len(postcode_files))
    print("First files:", postcode_files[:10])

    codepoint_cols = [
        "postcode",
        "positional_quality_indicator",
        "easting",
        "northing",
        "country_code",
        "nhs_regional_ha_code",
        "nhs_ha_code",
        "admin_county_code",
        "admin_district_code",
        "admin_ward_code"
    ]

    postcode_dfs = []

    for f in postcode_files:
        df = pd.read_csv(f, header=None, names=codepoint_cols)
        df["postcode_clean"] = df["postcode"].apply(clean_postcode)
        postcode_dfs.append(df[["postcode", "postcode_clean", "easting", "northing"]])

    postcodes = pd.concat(postcode_dfs, ignore_index=True)
    postcodes = postcodes.dropna(subset=["postcode_clean", "easting", "northing"])
    postcodes = postcodes.drop_duplicates(subset=["postcode_clean"])

    postcode_gdf = gpd.GeoDataFrame(
        postcodes,
        geometry=gpd.points_from_xy(postcodes["easting"], postcodes["northing"]),
        crs="EPSG:27700"
    )

    postcode_joined = gpd.sjoin(
        postcode_gdf,
        boroughs,
        how="left",
        predicate="within"
    )

    postcode_joined = postcode_joined.dropna(subset=["borough"]).copy()
    postcode_joined["x"] = postcode_joined.geometry.x
    postcode_joined["y"] = postcode_joined.geometry.y

    postcode_wgs84 = postcode_joined.to_crs(epsg=4326)
    postcode_joined["lon"] = postcode_wgs84.geometry.x
    postcode_joined["lat"] = postcode_wgs84.geometry.y

    print("London postcode coordinates:", postcode_joined.shape)
    display(postcode_joined.head())
else:
    print("Skipping postcode raw rebuild.")

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Inspecting EPC columns")

    epc_preview = pd.read_csv(EPC_FILE, nrows=5, low_memory=False)
    print("EPC preview columns:")
    print(epc_preview.columns.tolist())
    display(epc_preview.head())
else:
    print("Skipping EPC column preview.")

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Processing EPC certificates in chunks")

    epc_columns = epc_preview.columns

    postcode_col = pick_column(epc_columns, ["POSTCODE", "postcode"])
    eff_col = pick_column(epc_columns, [
        "CURRENT_ENERGY_EFFICIENCY",
        "current-energy-efficiency",
        "CURRENT-ENERGY-EFFICIENCY"
    ])
    rating_col = pick_column(epc_columns, [
        "CURRENT_ENERGY_RATING",
        "current-energy-rating",
        "CURRENT-ENERGY-RATING"
    ])

    property_type_col = pick_column(epc_columns, ["PROPERTY_TYPE", "property-type", "PROPERTY-TYPE"], required=False)
    built_form_col = pick_column(epc_columns, ["BUILT_FORM", "built-form", "BUILT-FORM"], required=False)
    inspection_col = pick_column(epc_columns, ["INSPECTION_DATE", "inspection-date", "INSPECTION-DATE"], required=False)
    lodgement_col = pick_column(epc_columns, ["LODGEMENT_DATE", "lodgement-date", "LODGEMENT-DATE"], required=False)
    local_auth_col = pick_column(epc_columns, ["LOCAL_AUTHORITY", "local-authority", "LOCAL-AUTHORITY"], required=False)
    uprn_col = pick_column(epc_columns, ["UPRN", "uprn"], required=False)

    usecols = [postcode_col, eff_col, rating_col]
    optional_cols = [
        property_type_col,
        built_form_col,
        inspection_col,
        lodgement_col,
        local_auth_col,
        uprn_col
    ]

    for c in optional_cols:
        if c is not None and c not in usecols:
            usecols.append(c)

    print("Using EPC columns:", usecols)

    chunks = []
    chunk_size = 300_000

    for i, chunk in enumerate(pd.read_csv(EPC_FILE, usecols=usecols, chunksize=chunk_size, low_memory=False)):
        rename_map = {
            postcode_col: "postcode",
            eff_col: "current_energy_efficiency",
            rating_col: "current_energy_rating"
        }

        if property_type_col is not None:
            rename_map[property_type_col] = "property_type"
        if built_form_col is not None:
            rename_map[built_form_col] = "built_form"
        if inspection_col is not None:
            rename_map[inspection_col] = "inspection_date"
        if lodgement_col is not None:
            rename_map[lodgement_col] = "lodgement_date"
        if local_auth_col is not None:
            rename_map[local_auth_col] = "local_authority"
        if uprn_col is not None:
            rename_map[uprn_col] = "uprn"

        chunk = chunk.rename(columns=rename_map)

        for col in [
            "property_type",
            "built_form",
            "inspection_date",
            "lodgement_date",
            "local_authority",
            "uprn"
        ]:
            if col not in chunk.columns:
                chunk[col] = np.nan

        chunk["postcode_clean"] = chunk["postcode"].apply(clean_postcode)
        chunk["current_energy_efficiency"] = pd.to_numeric(
            chunk["current_energy_efficiency"],
            errors="coerce"
        )

        chunk = chunk.dropna(subset=["postcode_clean", "current_energy_efficiency"])

        chunk = chunk[
            [
                "postcode_clean",
                "uprn",
                "current_energy_efficiency",
                "current_energy_rating",
                "property_type",
                "built_form",
                "inspection_date",
                "lodgement_date",
                "local_authority"
            ]
        ]

        chunks.append(chunk)

        if i % 5 == 0:
            rows_so_far = sum(len(c) for c in chunks)
            print(f"Processed chunk {i}; rows kept so far: {rows_so_far:,}")

    epc_small = pd.concat(chunks, ignore_index=True)
    print("EPC rows after basic cleaning:", epc_small.shape)
    display(epc_small.head())
else:
    print("Skipping EPC raw chunk processing.")

When UPRN is available, the notebook keeps the latest certificate per property before aggregating to postcode.  
This avoids giving extra weight to properties with multiple EPC records. If UPRN is not available, the workflow falls back to postcode-level aggregation from all usable certificates.

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Preparing latest EPC records and postcode-level labels")

    epc_small["inspection_date_parsed"] = pd.to_datetime(epc_small["inspection_date"], errors="coerce")
    epc_small["lodgement_date_parsed"] = pd.to_datetime(epc_small["lodgement_date"], errors="coerce")
    epc_small["record_date"] = epc_small["inspection_date_parsed"].fillna(epc_small["lodgement_date_parsed"])

    has_uprn = epc_small["uprn"].notna().any() and (epc_small["uprn"].astype(str).str.strip() != "").any()
    print("UPRN available:", has_uprn)

    if has_uprn:
        before = len(epc_small)
        epc_latest = (
            epc_small
            .sort_values("record_date")
            .drop_duplicates(subset=["uprn"], keep="last")
            .copy()
        )
        print(f"Kept latest record per UPRN: {before:,} -> {len(epc_latest):,}")
    else:
        epc_latest = epc_small.copy()
        print("No usable UPRN found. Aggregating all valid EPC records to postcode level.")

    # Show possible outliers before postcode aggregation.
    high_eff_count = (epc_latest["current_energy_efficiency"] > 100).sum()
    low_eff_count = (epc_latest["current_energy_efficiency"] < 1).sum()
    print("Raw latest EPC records with efficiency > 100:", int(high_eff_count))
    print("Raw latest EPC records with efficiency < 1:", int(low_eff_count))

    epc_by_postcode = (
        epc_latest
        .groupby("postcode_clean")
        .agg(
            current_energy_efficiency=("current_energy_efficiency", "mean"),
            current_energy_rating=("current_energy_rating", mode_or_nan),
            property_type=("property_type", mode_or_nan),
            built_form=("built_form", mode_or_nan),
            local_authority=("local_authority", mode_or_nan),
            n_certificates=("current_energy_efficiency", "size")
        )
        .reset_index()
    )

    print("Postcode-level EPC records before coordinate join:", epc_by_postcode.shape)
    display(epc_by_postcode.head())

    postcode_coords = postcode_joined[
        ["postcode_clean", "postcode", "x", "y", "lon", "lat", "borough"]
    ].copy()

    epc_joined = epc_by_postcode.merge(
        postcode_coords,
        on="postcode_clean",
        how="inner"
    )

    print("EPC postcodes joined with London coordinates:", epc_joined.shape)

    epc_samples = pd.DataFrame({
        "sample_id": ["EPC_" + str(i).zfill(7) for i in range(len(epc_joined))],
        "task": "EPC",
        "x": epc_joined["x"].values,
        "y": epc_joined["y"].values,
        "lon": epc_joined["lon"].values,
        "lat": epc_joined["lat"].values,
        "borough": epc_joined["borough"].values,
        "postcode": epc_joined["postcode"].values,
        "postcode_clean": epc_joined["postcode_clean"].values,
        "label_regression": epc_joined["current_energy_efficiency"].values,
        "label_classification": epc_joined["current_energy_rating"].astype(str).values,
        "property_type": epc_joined["property_type"].values,
        "built_form": epc_joined["built_form"].values,
        "n_certificates": epc_joined["n_certificates"].values,
        "crop_size_m": 150
    })

    epc_samples["epc_level"] = epc_samples["label_classification"].apply(epc_to_level)

    print("EPC sample table:", epc_samples.shape)
    display(epc_samples.head())
else:
    print("Skipping EPC sample table build.")

In [ ]:
if REBUILD_SAMPLE_TABLES:
    display_section("Saving raw sample tables")

    # Make PTAL and EPC columns consistent before combining.
    ptal_master = ptal_samples.copy()
    ptal_master["postcode"] = np.nan
    ptal_master["postcode_clean"] = np.nan
    ptal_master["property_type"] = np.nan
    ptal_master["built_form"] = np.nan
    ptal_master["n_certificates"] = np.nan
    ptal_master["epc_level"] = np.nan

    epc_master = epc_samples.copy()
    epc_master["ptal_level"] = np.nan

    common_cols = [
        "sample_id",
        "task",
        "x",
        "y",
        "lon",
        "lat",
        "borough",
        "postcode",
        "postcode_clean",
        "label_regression",
        "label_classification",
        "ptal_level",
        "epc_level",
        "property_type",
        "built_form",
        "n_certificates",
        "crop_size_m"
    ]

    sample_master = pd.concat(
        [
            ptal_master[common_cols],
            epc_master[common_cols]
        ],
        ignore_index=True
    )

    ptal_out = TABLE_DIR / "ptal_samples_raw.csv"
    epc_out = TABLE_DIR / "epc_samples_raw.csv"
    master_out = TABLE_DIR / "sample_master_raw.csv"

    ptal_samples.to_csv(ptal_out, index=False)
    epc_samples.to_csv(epc_out, index=False)
    sample_master.to_csv(master_out, index=False)

    print("Saved:")
    print(" -", ptal_out)
    print(" -", epc_out)
    print(" -", master_out)

    print("\nTask counts:")
    print(sample_master["task"].value_counts())
else:
    sample_master = pd.read_csv(TABLE_DIR / "sample_master_raw.csv")
    print("Loaded existing raw sample master:", sample_master.shape)

## 4. Clean the sample tables

The first cleaning step fixes the two main issues found in the initial run:

- PTAL points outside the borough boundary are removed, because borough-based validation needs a borough label.
- EPC postcode records with unusual energy efficiency values outside the 1–100 range are removed for the first modelling baseline.

In [ ]:
display_section("Cleaning sample tables")

ptal_clean = sample_master[sample_master["task"] == "PTAL"].copy()
epc_clean = sample_master[sample_master["task"] == "EPC"].copy()

print("Before cleaning:")
print("PTAL rows:", len(ptal_clean))
print("EPC rows:", len(epc_clean))
print("PTAL missing borough:", ptal_clean["borough"].isna().sum())
print("EPC missing borough:", epc_clean["borough"].isna().sum())

# PTAL cleaning
ptal_clean["label_regression"] = pd.to_numeric(ptal_clean["label_regression"], errors="coerce")
ptal_clean["ptal_level"] = ptal_clean["label_classification"].apply(ptal_to_level)
ptal_clean = ptal_clean.dropna(subset=["borough", "label_regression", "ptal_level"]).copy()

# EPC cleaning
epc_clean["label_regression"] = pd.to_numeric(epc_clean["label_regression"], errors="coerce")
epc_clean["epc_level"] = epc_clean["label_classification"].apply(epc_to_level)

epc_outlier_mask = (
    (epc_clean["label_regression"] < 1) |
    (epc_clean["label_regression"] > 100)
)
print("\nEPC records outside 1-100:", int(epc_outlier_mask.sum()))

display(epc_clean.loc[epc_outlier_mask].head(10))

epc_clean = epc_clean[
    (~epc_outlier_mask) &
    epc_clean["borough"].notna() &
    epc_clean["label_regression"].notna() &
    epc_clean["epc_level"].notna()
].copy()

sample_master_clean = pd.concat([ptal_clean, epc_clean], ignore_index=True)

print("\nAfter cleaning:")
print("PTAL rows:", len(ptal_clean))
print("EPC rows:", len(epc_clean))
print("Total rows:", len(sample_master_clean))

display(sample_master_clean.head())

In [ ]:
# Save the cleaned core tables for the embedding and modelling stages.
ptal_clean_out = TABLE_DIR / "ptal_samples_clean.csv"
epc_clean_out = TABLE_DIR / "epc_samples_clean.csv"
master_clean_out = TABLE_DIR / "sample_master_clean.csv"

ptal_clean.to_csv(ptal_clean_out, index=False)
epc_clean.to_csv(epc_clean_out, index=False)
sample_master_clean.to_csv(master_clean_out, index=False)

print("Saved cleaned tables:")
print(" -", ptal_clean_out)
print(" -", epc_clean_out)
print(" -", master_clean_out)

**Clean sample tables**  
These are the samples I actually use for modelling. PTAL points without a borough are removed because the spatial validation needs borough labels. EPC records with energy efficiency outside 1–100 are removed as obvious outliers.  



## 5. Quick descriptive checks

These checks make sure the labels have reasonable distributions before modelling.  
The charts are only displayed inside the notebook.

In [ ]:
display_section("Label summaries")

print("PTAL Access Index:")
display(ptal_clean["label_regression"].describe())

print("\nPTAL grouped classes:")
display(ptal_clean["ptal_level"].value_counts(dropna=False).to_frame("count"))

print("\nPTAL original bands:")
display(ptal_clean["label_classification"].value_counts(dropna=False).to_frame("count"))

print("\nEPC energy efficiency:")
display(epc_clean["label_regression"].describe())

print("\nEPC grouped classes:")
display(epc_clean["epc_level"].value_counts(dropna=False).to_frame("count"))

print("\nEPC original ratings:")
display(epc_clean["label_classification"].value_counts(dropna=False).to_frame("count"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ptal_clean["label_regression"].hist(bins=50, ax=ax)
ax.set_title("PTAL Access Index distribution")
ax.set_xlabel("Access Index")
ax.set_ylabel("Count")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
epc_clean["label_regression"].hist(bins=50, ax=ax)
ax.set_title("EPC energy efficiency distribution")
ax.set_xlabel("Energy efficiency score")
ax.set_ylabel("Count")
plt.show()

PTAL is strongly skewed: many places have low or medium accessibility, while very high-accessibility areas are fewer. EPC is also imbalanced, with most postcodes around the middle energy bands.  



## 6. Coordinate-only baseline models

The coordinate baseline is the first reality check.  
It tells us how much of each target can be predicted from location alone before adding aerial, satellite, or learned location embeddings.

Two validation settings are used:

- **Random split**: useful as a quick benchmark, but it can overestimate performance.
- **Borough GroupKFold**: a stricter spatial validation where test data comes from boroughs not used in training.

In [ ]:
def evaluate_regression_coordinate(df, target_col, task_name, groups_col="borough"):
    data = df.dropna(subset=["x", "y", target_col, groups_col]).copy()
    X = data[["x", "y"]].values
    y = data[target_col].values
    groups = data[groups_col].values

    models = {
        "Dummy mean": DummyRegressor(strategy="mean"),
        "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
        "HistGradientBoosting": HistGradientBoostingRegressor(
            max_iter=150,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        )
    }

    rows = []

    # Random split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE
    )

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            "task": task_name,
            "target": target_col,
            "problem": "regression",
            "split": "random_70_30",
            "model": model_name,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "RMSE": rmse(y_test, pred),
            "MAE": mean_absolute_error(y_test, pred),
            "R2": r2_score(y_test, pred)
        })

    # Borough GroupKFold
    gkf = GroupKFold(n_splits=N_SPLITS)

    for model_name, model in models.items():
        fold_metrics = []
        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model.fit(X_train, y_train)
            pred = model.predict(X_test)

            fold_metrics.append({
                "RMSE": rmse(y_test, pred),
                "MAE": mean_absolute_error(y_test, pred),
                "R2": r2_score(y_test, pred),
                "n_train": len(y_train),
                "n_test": len(y_test)
            })

        fold_df = pd.DataFrame(fold_metrics)
        rows.append({
            "task": task_name,
            "target": target_col,
            "problem": "regression",
            "split": f"borough_groupkfold_{N_SPLITS}",
            "model": model_name,
            "n_train": int(fold_df["n_train"].mean()),
            "n_test": int(fold_df["n_test"].mean()),
            "RMSE": fold_df["RMSE"].mean(),
            "MAE": fold_df["MAE"].mean(),
            "R2": fold_df["R2"].mean()
        })

    return pd.DataFrame(rows)


def evaluate_classification_coordinate(df, target_col, task_name, groups_col="borough"):
    data = df.dropna(subset=["x", "y", target_col, groups_col]).copy()
    X = data[["x", "y"]].values
    y = data[target_col].astype(str).values
    groups = data[groups_col].values

    models = {
        "Dummy most frequent": DummyClassifier(strategy="most_frequent"),
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        ),
        "HistGradientBoosting": HistGradientBoostingClassifier(
            max_iter=150,
            learning_rate=0.05,
            random_state=RANDOM_STATE
        )
    }

    rows = []

    stratify = y if pd.Series(y).value_counts().min() >= 2 else None

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.3,
        random_state=RANDOM_STATE,
        stratify=stratify
    )

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        rows.append({
            "task": task_name,
            "target": target_col,
            "problem": "classification",
            "split": "random_70_30",
            "model": model_name,
            "n_train": len(y_train),
            "n_test": len(y_test),
            "accuracy": accuracy_score(y_test, pred),
            "macro_f1": f1_score(y_test, pred, average="macro")
        })

    gkf = GroupKFold(n_splits=N_SPLITS)

    for model_name, model in models.items():
        fold_metrics = []
        for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            model.fit(X_train, y_train)
            pred = model.predict(X_test)

            fold_metrics.append({
                "accuracy": accuracy_score(y_test, pred),
                "macro_f1": f1_score(y_test, pred, average="macro"),
                "n_train": len(y_train),
                "n_test": len(y_test)
            })

        fold_df = pd.DataFrame(fold_metrics)
        rows.append({
            "task": task_name,
            "target": target_col,
            "problem": "classification",
            "split": f"borough_groupkfold_{N_SPLITS}",
            "model": model_name,
            "n_train": int(fold_df["n_train"].mean()),
            "n_test": int(fold_df["n_test"].mean()),
            "accuracy": fold_df["accuracy"].mean(),
            "macro_f1": fold_df["macro_f1"].mean()
        })

    return pd.DataFrame(rows)

In [ ]:
display_section("Running coordinate baselines")

baseline_results = []

baseline_results.append(
    evaluate_regression_coordinate(
        ptal_clean,
        target_col="label_regression",
        task_name="PTAL Access Index"
    )
)

baseline_results.append(
    evaluate_classification_coordinate(
        ptal_clean,
        target_col="ptal_level",
        task_name="PTAL level"
    )
)

baseline_results.append(
    evaluate_regression_coordinate(
        epc_clean,
        target_col="label_regression",
        task_name="EPC energy efficiency"
    )
)

baseline_results.append(
    evaluate_classification_coordinate(
        epc_clean,
        target_col="epc_level",
        task_name="EPC level"
    )
)

baseline_results = pd.concat(baseline_results, ignore_index=True)

display(baseline_results)

In [ ]:
baseline_out = TABLE_DIR / "coordinate_baseline_results.csv"
baseline_results.to_csv(baseline_out, index=False)
print("Saved baseline results:", baseline_out)

**Coordinate baseline**  
This is the first reality check. It asks: *if the model only knows where a point is, how much can it predict?*  
For PTAL, location alone is already quite informative. For EPC, location is much weaker, so image or EO features may matter more.  



## 7. Full aerial crop index

This index records the square crop extent for every cleaned sample.  
It does not create image chips yet. It only records where each chip should be cropped later.

- PTAL uses a 300m × 300m square.
- EPC uses a 150m × 150m square.

In [ ]:
display_section("Creating full aerial crop index")

crop_index = sample_master_clean[
    ["sample_id", "task", "x", "y", "borough", "label_regression", "label_classification", "crop_size_m"]
].copy()

crop_index["half_size_m"] = crop_index["crop_size_m"] / 2
crop_index["xmin"] = crop_index["x"] - crop_index["half_size_m"]
crop_index["xmax"] = crop_index["x"] + crop_index["half_size_m"]
crop_index["ymin"] = crop_index["y"] - crop_index["half_size_m"]
crop_index["ymax"] = crop_index["y"] + crop_index["half_size_m"]

crop_index["output_path"] = crop_index.apply(
    lambda row: f"outputs/chips/{row['task'].lower()}_{int(row['crop_size_m'])}m/{row['sample_id']}.jpg",
    axis=1
)

crop_index = crop_index.drop(columns=["half_size_m"])

display(crop_index.head())
print(crop_index["task"].value_counts())

crop_full_out = TABLE_DIR / "aerial_crop_index_full_clean.csv"
crop_index.to_csv(crop_full_out, index=False)
print("Saved full crop index:", crop_full_out)

## 8. London-wide DINO/aerial sample index

The full cleaned dataset contains hundreds of thousands of potential image chips.  
For the first DINOv3 extraction run, the notebook creates a smaller but London-wide sample:

- **PTAL**: one representative point per 500m grid cell.
- **EPC**: a stratified postcode sample by borough and EPC level.

This keeps London-wide spatial coverage while making the first aerial embedding run manageable.

In [ ]:
def nearest_to_cell_centre(group, xbin_col, ybin_col, grid_size):
    row = group.iloc[0]
    cx = (row[xbin_col] + 0.5) * grid_size
    cy = (row[ybin_col] + 0.5) * grid_size
    dist = (group["x"] - cx) ** 2 + (group["y"] - cy) ** 2
    return group.loc[[dist.idxmin()]]


def make_ptal_systematic_sample(ptal_df, grid_size_m=500):
    df = ptal_df.copy()
    df["grid_x"] = np.floor(df["x"] / grid_size_m).astype(int)
    df["grid_y"] = np.floor(df["y"] / grid_size_m).astype(int)

    sampled = (
        df
        .groupby(["grid_x", "grid_y"], group_keys=False)
        .apply(lambda g: nearest_to_cell_centre(g, "grid_x", "grid_y", grid_size_m))
        .drop(columns=["grid_x", "grid_y"])
        .reset_index(drop=True)
    )

    return sampled


def stratified_sample_by_group(df, strata_cols, target_n, random_state=42, min_per_group=5):
    df = df.copy()
    df["_strata"] = df[strata_cols].astype(str).agg("|".join, axis=1)

    n_total = len(df)

    sampled_parts = []
    for strata, group in df.groupby("_strata"):
        proportional_n = int(round(target_n * len(group) / n_total))
        n = max(min_per_group, proportional_n)
        n = min(n, len(group))
        sampled_parts.append(group.sample(n=n, random_state=random_state))

    sampled = pd.concat(sampled_parts, ignore_index=True)

    # If the sample is larger than target because of min_per_group, downsample while preserving broad strata.
    if len(sampled) > target_n:
        sampled = sampled.sample(n=target_n, random_state=random_state).reset_index(drop=True)

    sampled = sampled.drop(columns=["_strata"])
    return sampled

In [ ]:
display_section("Creating DINO/aerial sample index")

ptal_dino_sample = make_ptal_systematic_sample(
    ptal_clean,
    grid_size_m=PTAL_DINO_GRID_M
)

epc_dino_sample = stratified_sample_by_group(
    epc_clean,
    strata_cols=["borough", "epc_level"],
    target_n=EPC_DINO_TARGET_N,
    random_state=RANDOM_STATE,
    min_per_group=5
)

dino_samples = pd.concat([ptal_dino_sample, epc_dino_sample], ignore_index=True)

print("PTAL DINO sample:", len(ptal_dino_sample))
print("EPC DINO sample:", len(epc_dino_sample))
print("Total DINO sample:", len(dino_samples))

display(dino_samples[["sample_id", "task", "x", "y", "borough", "label_regression", "label_classification", "crop_size_m"]].head())
display(dino_samples["task"].value_counts().to_frame("count"))

In [ ]:
dino_crop_index = dino_samples[
    ["sample_id", "task", "x", "y", "borough", "label_regression", "label_classification", "crop_size_m"]
].copy()

dino_crop_index["half_size_m"] = dino_crop_index["crop_size_m"] / 2
dino_crop_index["xmin"] = dino_crop_index["x"] - dino_crop_index["half_size_m"]
dino_crop_index["xmax"] = dino_crop_index["x"] + dino_crop_index["half_size_m"]
dino_crop_index["ymin"] = dino_crop_index["y"] - dino_crop_index["half_size_m"]
dino_crop_index["ymax"] = dino_crop_index["y"] + dino_crop_index["half_size_m"]

dino_crop_index["output_path"] = dino_crop_index.apply(
    lambda row: f"outputs/chips_dino_sample/{row['task'].lower()}_{int(row['crop_size_m'])}m/{row['sample_id']}.jpg",
    axis=1
)

dino_crop_index = dino_crop_index.drop(columns=["half_size_m"])

dino_samples_out = TABLE_DIR / "dino_londonwide_sample.csv"
dino_crop_out = TABLE_DIR / "aerial_crop_index_dino_londonwide_sample.csv"

dino_samples.to_csv(dino_samples_out, index=False)
dino_crop_index.to_csv(dino_crop_out, index=False)

print("Saved DINO sample table:", dino_samples_out)
print("Saved DINO crop index:", dino_crop_out)

display(dino_crop_index.head())

In [ ]:
# Simple spatial preview of the DINO/aerial sample.
preview = dino_samples.sample(min(10000, len(dino_samples)), random_state=RANDOM_STATE)

fig, ax = plt.subplots(figsize=(7, 7))
preview.plot.scatter(x="x", y="y", ax=ax, s=2)
ax.set_title("London-wide aerial sample preview")
ax.set_xlabel("British National Grid easting")
ax.set_ylabel("British National Grid northing")
plt.show()

**Aerial sample design**  
The full cleaned dataset is too large for aerial image extraction. I therefore create a London-wide but smaller sample for DINOv3: it still covers the whole city, but keeps the image workload manageable.  


In [ ]:
display_section("Final output summary")

outputs_to_check = [
    ptal_clean_out,
    epc_clean_out,
    master_clean_out,
    baseline_out,
    crop_full_out,
    dino_samples_out,
    dino_crop_out
]

for p in outputs_to_check:
    print(f"{p.name}: exists = {p.exists()} | path = {p}")

print("\nCleaned sample counts:")
print(sample_master_clean["task"].value_counts())

print("\nDINO sample counts:")
print(dino_samples["task"].value_counts())

print("\nCoordinate baseline results:")
display(baseline_results)

**Checkpoint**  
By this point, the London sample framework is ready: cleaned PTAL and EPC tables, coordinate baselines, full crop index, and a smaller London-wide image sample for TESSERA and DINOv3 comparison.  


## SatCLIP location embeddings

This step extracts SatCLIP embeddings for all cleaned PTAL and EPC samples, then tests whether the learned location representation improves downstream prediction compared with raw coordinates.

In [ ]:
# ============================================================
# SatCLIP embeddings: clean setup, extract once, then reuse
# ============================================================

!pip -q install huggingface_hub pyarrow pytorch-lightning lightning einops torchgeo timm rasterio

from pathlib import Path
import sys
import shutil
import subprocess
import pandas as pd
import numpy as np
import torch
from huggingface_hub import hf_hub_download

BASE = Path("/content/drive/MyDrive/GEOG0105")
TABLE_DIR = BASE / "Outputs" / "tables"
EMB_DIR = BASE / "Outputs" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

sample_master = pd.read_csv(TABLE_DIR / "sample_master_clean.csv")
satclip_path = EMB_DIR / "satclip_embeddings_full.parquet"

if satclip_path.exists():
    satclip_embeddings = pd.read_parquet(satclip_path)
    print("Loaded existing SatCLIP embeddings:", satclip_embeddings.shape)

else:
    repo = Path("/content/satclip")

    # Remove the previous broken repo and clone a clean copy.
    if repo.exists():
        shutil.rmtree(repo)

    subprocess.run(
        ["git", "clone", "-q", "https://github.com/microsoft/satclip.git", str(repo)],
        check=True
    )

    sys.path.insert(0, str(repo))
    sys.path.insert(0, str(repo / "satclip"))

    # Patch TorchGeo import safely.
    patch_file = repo / "satclip" / "datamodules" / "s2geo_dataset.py"
    txt = patch_file.read_text()

    old_line = "from torchgeo.datasets.geo import NonGeoDataset"
    new_block = (
        "try:\n"
        "    from torchgeo.datasets.geo import NonGeoDataset\n"
        "except Exception:\n"
        "    try:\n"
        "        from torchgeo.datasets import NonGeoDataset\n"
        "    except Exception:\n"
        "        from torch.utils.data import Dataset as NonGeoDataset\n"
    )

    if old_line in txt:
        txt = txt.replace(old_line, new_block)
        patch_file.write_text(txt)
        print("Patched TorchGeo import.")
    else:
        print("TorchGeo import line not found; patch skipped.")

    # Clear cached imports if the notebook has previously failed.
    for module_name in list(sys.modules):
        if module_name == "load" or module_name.startswith("satclip"):
            del sys.modules[module_name]

    from load import get_satclip

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    ckpt = hf_hub_download(
        repo_id="microsoft/SatCLIP-ResNet18-L40",
        filename="satclip-resnet18-l40.ckpt"
    )

    model = get_satclip(ckpt, device=device).eval()

    coords = sample_master[["lon", "lat"]].to_numpy(dtype=np.float64)
    sample_ids = sample_master["sample_id"].to_numpy()

    batch_size = 8192
    emb_list = []

    with torch.no_grad():
        for start in range(0, len(coords), batch_size):
            end = min(start + batch_size, len(coords))
            batch = torch.tensor(coords[start:end], dtype=torch.float64, device=device)

            emb = model(batch).detach().cpu().numpy().astype(np.float32)
            emb_list.append(emb)

            if start == 0 or (start // batch_size) % 10 == 0:
                print(f"Processed {end:,} / {len(coords):,}")

    emb_array = np.vstack(emb_list)

    satclip_embeddings = pd.DataFrame(
        emb_array,
        columns=[f"satclip_{i:03d}" for i in range(emb_array.shape[1])]
    )
    satclip_embeddings.insert(0, "sample_id", sample_ids)

    satclip_embeddings.to_parquet(satclip_path, index=False)
    print("Saved SatCLIP embeddings:", satclip_path)

print("SatCLIP embeddings ready:", satclip_embeddings.shape)
display(satclip_embeddings.head())

**SatCLIP embedding**  
SatCLIP takes longitude and latitude and turns each location into a 256-dimensional learned location embedding. It is still a location-based feature, but stronger than raw x-y coordinates because it comes from pretraining.  


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    return {
        "rmse": rmse,
        "mae": mean_absolute_error(y_true, y_pred),
        "r2": r2_score(y_true, y_pred)
    }

def classification_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

print("Metric functions updated.")

In [ ]:
# ============================================================
# Evaluate SatCLIP embeddings on PTAL and EPC
# ============================================================

from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeClassifier

needed = [
    "evaluate_regression_random",
    "evaluate_regression_groupkfold",
    "evaluate_classification_random",
    "evaluate_classification_groupkfold"
]

missing = [f for f in needed if f not in globals()]
if missing:
    raise NameError(f"Rerun the earlier modelling-functions cell first. Missing: {missing}")


def run_embedding_baseline(embedding_df, feature_prefix, feature_set_name):
    feature_cols = [c for c in embedding_df.columns if c.startswith(feature_prefix)]

    data = sample_master.merge(
        embedding_df[["sample_id"] + feature_cols],
        on="sample_id",
        how="inner"
    )

    results = []

    for task_name, class_col in [("PTAL", "ptal_level"), ("EPC", "epc_level")]:
        df = data[data["task"] == task_name].dropna(subset=["borough"]).copy()

        reg_df = df.dropna(subset=["label_regression"])
        X_reg = reg_df[feature_cols].to_numpy(dtype=np.float32)
        y_reg = reg_df["label_regression"].to_numpy(dtype=np.float32)
        g_reg = reg_df["borough"].to_numpy()

        reg_models = {
            "DummyMean": DummyRegressor(strategy="mean"),
            f"{feature_set_name}_Ridge": Pipeline([
                ("scaler", StandardScaler()),
                ("ridge", Ridge(alpha=1.0))
            ])
        }

        for model_name, model in reg_models.items():
            results.append(evaluate_regression_random(
                X_reg, y_reg, model, model_name, task_name, feature_set_name
            ))
            results.append(evaluate_regression_groupkfold(
                X_reg, y_reg, g_reg, model, model_name, task_name, feature_set_name
            ))

        cls_df = df.dropna(subset=[class_col])
        X_cls = cls_df[feature_cols].to_numpy(dtype=np.float32)
        y_cls = cls_df[class_col].astype(str).to_numpy()
        g_cls = cls_df["borough"].to_numpy()

        cls_models = {
            "DummyMostFrequent": DummyClassifier(strategy="most_frequent"),
            f"{feature_set_name}_RidgeClassifier": Pipeline([
                ("scaler", StandardScaler()),
                ("ridge_cls", RidgeClassifier(class_weight="balanced"))
            ])
        }

        for model_name, model in cls_models.items():
            results.append(evaluate_classification_random(
                X_cls, y_cls, model, model_name, task_name, feature_set_name
            ))
            results.append(evaluate_classification_groupkfold(
                X_cls, y_cls, g_cls, model, model_name, task_name, feature_set_name
            ))

    return pd.DataFrame(results)


satclip_results = run_embedding_baseline(
    embedding_df=satclip_embeddings,
    feature_prefix="satclip_",
    feature_set_name="SatCLIP"
)

satclip_results.to_csv(TABLE_DIR / "satclip_baseline_results.csv", index=False)

display(satclip_results)
print("Saved:", TABLE_DIR / "satclip_baseline_results.csv")

In [ ]:
# ============================================================
# Compare coordinate baseline and SatCLIP baseline
# ============================================================

coord_path = TABLE_DIR / "coordinate_baseline_results.csv"

if coord_path.exists():
    coord_results = pd.read_csv(coord_path)
    comparison_results = pd.concat([coord_results, satclip_results], ignore_index=True)
else:
    comparison_results = satclip_results.copy()

print("Regression results:")
display(
    comparison_results[comparison_results["target_type"] == "regression"]
    .sort_values(["task", "validation", "rmse"])
)

print("Classification results:")
display(
    comparison_results[comparison_results["target_type"] == "classification"]
    .sort_values(["task", "validation", "macro_f1"], ascending=[True, True, False])
)

In [ ]:
# ============================================================
# Clean and combine baseline result tables
# ============================================================

coord_results = pd.read_csv(TABLE_DIR / "coordinate_baseline_results.csv")
satclip_results = pd.read_csv(TABLE_DIR / "satclip_baseline_results.csv")

# Standardise metric column names from older coordinate result table
rename_map = {
    "RMSE": "rmse",
    "MAE": "mae",
    "R2": "r2"
}

coord_results = coord_results.rename(columns=rename_map)
satclip_results = satclip_results.rename(columns=rename_map)

# Keep only useful common columns
keep_cols = [
    "task", "target_type", "feature_set", "validation", "model",
    "rmse", "mae", "r2", "accuracy", "macro_f1"
]

for col in keep_cols:
    if col not in coord_results.columns:
        coord_results[col] = np.nan
    if col not in satclip_results.columns:
        satclip_results[col] = np.nan

baseline_clean = pd.concat(
    [coord_results[keep_cols], satclip_results[keep_cols]],
    ignore_index=True
)

baseline_clean.to_csv(TABLE_DIR / "baseline_results_clean.csv", index=False)

print("Clean regression results:")
display(
    baseline_clean[baseline_clean["target_type"] == "regression"]
    .sort_values(["task", "validation", "rmse"])
)

print("Clean classification results:")
display(
    baseline_clean[baseline_clean["target_type"] == "classification"]
    .sort_values(["task", "validation", "macro_f1"], ascending=[True, True, False])
)

**SatCLIP result**  
SatCLIP helps PTAL to some extent, especially compared with a dummy model, but it is weak for EPC. This suggests SatCLIP is useful as a learned location baseline, but not enough for the housing-related task.  


## Combined coordinate and SatCLIP baseline

This step tests whether the SatCLIP location embedding adds useful information beyond raw geographic coordinates. To keep the comparison fair and simple, the same linear models are applied to three feature sets: raw coordinates only, SatCLIP only, and coordinates combined with SatCLIP.

The evaluation focuses on PTAL regression, PTAL classification, and EPC regression. EPC classification is not used here because the grouped EPC classes are highly imbalanced and the previous results showed that accuracy and macro-F1 were not very informative for this task.

In [ ]:
# Coordinates + SatCLIP combined baseline


# Load data
sample_master = pd.read_csv(TABLE_DIR / "sample_master_clean.csv")
satclip_embeddings = pd.read_parquet(EMB_DIR / "satclip_embeddings_full.parquet")

data = sample_master.merge(satclip_embeddings, on="sample_id", how="inner")

coord_cols = ["x", "y"]
satclip_cols = [c for c in data.columns if c.startswith("satclip_")]

feature_sets = {
    "Coord_xy": coord_cols,
    "SatCLIP": satclip_cols,
    "Coord_xy_SatCLIP": coord_cols + satclip_cols
}

def reg_metrics(y_true, pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, pred)),
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred)
    }

def cls_metrics(y_true, pred):
    return {
        "accuracy": accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro")
    }

def eval_random_regression(X, y, model):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )
    model.fit(X_train, y_train)
    return reg_metrics(y_test, model.predict(X_test))

def eval_group_regression(X, y, groups, model, n_splits=5):
    rows = []
    gkf = GroupKFold(n_splits=n_splits)

    for train_idx, test_idx in gkf.split(X, y, groups):
        model.fit(X[train_idx], y[train_idx])
        rows.append(reg_metrics(y[test_idx], model.predict(X[test_idx])))

    return pd.DataFrame(rows).mean().to_dict()

def eval_random_classification(X, y, model):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    return cls_metrics(y_test, model.predict(X_test))

def eval_group_classification(X, y, groups, model, n_splits=5):
    rows = []
    gkf = GroupKFold(n_splits=n_splits)

    for train_idx, test_idx in gkf.split(X, y, groups):
        model.fit(X[train_idx], y[train_idx])
        rows.append(cls_metrics(y[test_idx], model.predict(X[test_idx])))

    return pd.DataFrame(rows).mean().to_dict()

def make_reg_model(model_type):
    if model_type == "Dummy":
        return DummyRegressor(strategy="mean")
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

def make_cls_model(model_type):
    if model_type == "Dummy":
        return DummyClassifier(strategy="most_frequent")
    return Pipeline([
        ("scaler", StandardScaler()),
        ("ridge_classifier", RidgeClassifier(class_weight="balanced"))
    ])

results = []

# -----------------------------
# 1. PTAL regression
# -----------------------------
ptal = data[data["task"] == "PTAL"].dropna(subset=["label_regression", "borough"]).copy()
y = ptal["label_regression"].to_numpy(dtype=np.float32)
groups = ptal["borough"].to_numpy()

for feature_name, cols in feature_sets.items():
    X = ptal[cols].to_numpy(dtype=np.float32)

    for model_name in ["Dummy", "Ridge"]:
        model = make_reg_model(model_name)

        for validation, metrics in [
            ("random_70_30", eval_random_regression(X, y, model)),
            ("borough_groupkfold_5", eval_group_regression(X, y, groups, model))
        ]:
            results.append({
                "task": "PTAL",
                "target_type": "regression",
                "feature_set": feature_name,
                "validation": validation,
                "model": model_name,
                **metrics
            })

# -----------------------------
# 2. PTAL classification
# -----------------------------
ptal_cls = data[data["task"] == "PTAL"].dropna(subset=["ptal_level", "borough"]).copy()
y = ptal_cls["ptal_level"].astype(str).to_numpy()
groups = ptal_cls["borough"].to_numpy()

for feature_name, cols in feature_sets.items():
    X = ptal_cls[cols].to_numpy(dtype=np.float32)

    for model_name in ["Dummy", "RidgeClassifier"]:
        model = make_cls_model(model_name)

        for validation, metrics in [
            ("random_70_30", eval_random_classification(X, y, model)),
            ("borough_groupkfold_5", eval_group_classification(X, y, groups, model))
        ]:
            results.append({
                "task": "PTAL",
                "target_type": "classification",
                "feature_set": feature_name,
                "validation": validation,
                "model": model_name,
                **metrics
            })

# -----------------------------
# 3. EPC regression
# -----------------------------
epc = data[data["task"] == "EPC"].dropna(subset=["label_regression", "borough"]).copy()
y = epc["label_regression"].to_numpy(dtype=np.float32)
groups = epc["borough"].to_numpy()

for feature_name, cols in feature_sets.items():
    X = epc[cols].to_numpy(dtype=np.float32)

    for model_name in ["Dummy", "Ridge"]:
        model = make_reg_model(model_name)

        for validation, metrics in [
            ("random_70_30", eval_random_regression(X, y, model)),
            ("borough_groupkfold_5", eval_group_regression(X, y, groups, model))
        ]:
            results.append({
                "task": "EPC",
                "target_type": "regression",
                "feature_set": feature_name,
                "validation": validation,
                "model": model_name,
                **metrics
            })

combined_baseline_results = pd.DataFrame(results)
combined_baseline_results.to_csv(TABLE_DIR / "coord_satclip_combined_results.csv", index=False)

print("Saved:", TABLE_DIR / "coord_satclip_combined_results.csv")

In [ ]:
# ============================================================
# Show key results
# ============================================================

res = combined_baseline_results.copy()

print("PTAL regression: lower RMSE and higher R² are better")
display(
    res[
        (res["task"] == "PTAL") &
        (res["target_type"] == "regression") &
        (res["model"] == "Ridge")
    ][["feature_set", "validation", "rmse", "mae", "r2"]]
    .sort_values(["validation", "rmse"])
)

print("PTAL classification: higher macro-F1 is better")
display(
    res[
        (res["task"] == "PTAL") &
        (res["target_type"] == "classification") &
        (res["model"] == "RidgeClassifier")
    ][["feature_set", "validation", "accuracy", "macro_f1"]]
    .sort_values(["validation", "macro_f1"], ascending=[True, False])
)

print("EPC regression: lower RMSE and higher R² are better")
display(
    res[
        (res["task"] == "EPC") &
        (res["target_type"] == "regression") &
        (res["model"] == "Ridge")
    ][["feature_set", "validation", "rmse", "mae", "r2"]]
    .sort_values(["validation", "rmse"])
)

**Coord + SatCLIP**  
Here I check whether SatCLIP adds information beyond raw coordinates. The result is quite simple: combining coordinates with SatCLIP gives little extra improvement over SatCLIP alone.  



## TESSERA satellite EO embeddings

This section adds TESSERA as a satellite Earth observation representation. TESSERA provides 128-dimensional pixel-level embeddings derived from Sentinel-1 and Sentinel-2 time-series observations. The embeddings are sampled at the same London-wide subset that will later be used for aerial image extraction, so the TESSERA and DINOv3 experiments can be compared on the same set of locations.

This step focuses on three tasks: PTAL regression, PTAL classification, and EPC regression. EPC classification is not included because the grouped EPC classes are highly imbalanced.

In [ ]:
# ============================================================
# TESSERA embeddings: test first, then full extraction
# ============================================================

!pip -q install geotessera pyarrow

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/GEOG0105")
TABLE_DIR = BASE / "Outputs" / "tables"
EMB_DIR = BASE / "Outputs" / "embeddings"
EMB_DIR.mkdir(parents=True, exist_ok=True)

sample_master = pd.read_csv(TABLE_DIR / "sample_master_clean.csv")
dino_sample = pd.read_csv(TABLE_DIR / "dino_londonwide_sample.csv")

# Use the same London-wide subset prepared for later DINOv3 extraction.
sample_ids = dino_sample["sample_id"].unique()
tessera_samples = sample_master[sample_master["sample_id"].isin(sample_ids)].copy()

print("TESSERA target sample:")
print(tessera_samples["task"].value_counts())
print("Total:", len(tessera_samples))

# Start with a small test first.
RUN_FULL_TESSERA = True   # Change to True after the test succeeds.
TEST_N = 300

if RUN_FULL_TESSERA:
    run_samples = tessera_samples.copy()
    out_path = EMB_DIR / "tessera_embeddings_dino_sample.parquet"
else:
    run_samples = tessera_samples.sample(
        n=min(TEST_N, len(tessera_samples)),
        random_state=42
    ).copy()
    out_path = EMB_DIR / "tessera_embeddings_test.parquet"

print("Running TESSERA sample size:", len(run_samples))
print("Output path:", out_path)

if out_path.exists():
    tessera_embeddings = pd.read_parquet(out_path)
    print("Loaded existing TESSERA embeddings:", tessera_embeddings.shape)

else:
    from geotessera import GeoTessera

    gt = GeoTessera()

    points = list(zip(run_samples["lon"], run_samples["lat"]))

    print("Sampling TESSERA embeddings. First run may take time because tiles are downloaded and cached...")
    emb = gt.sample_embeddings_at_points(points, year=2024)

    emb = np.asarray(emb, dtype=np.float32)

    tessera_embeddings = pd.DataFrame(
        emb,
        columns=[f"tessera_{i:03d}" for i in range(emb.shape[1])]
    )
    tessera_embeddings.insert(0, "sample_id", run_samples["sample_id"].to_numpy())

    tessera_embeddings.to_parquet(out_path, index=False)
    print("Saved TESSERA embeddings:", out_path)

print("TESSERA embeddings ready:", tessera_embeddings.shape)
display(tessera_embeddings.head())

## TESSERA baseline comparison

TESSERA embeddings have now been extracted for the London-wide image sample. This section compares TESSERA with the existing location-based representations on the same subset of samples.

The comparison focuses on PTAL regression, PTAL classification, and EPC regression. EPC classification is not included because the previous results showed strong class imbalance and weak interpretability.

In [ ]:
# ============================================================
# TESSERA comparison on the same London-wide subset
# ============================================================

from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, RidgeClassifier
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, f1_score

# Load required files
sample_master = pd.read_csv(TABLE_DIR / "sample_master_clean.csv")
satclip_embeddings = pd.read_parquet(EMB_DIR / "satclip_embeddings_full.parquet")
tessera_embeddings = pd.read_parquet(EMB_DIR / "tessera_embeddings_dino_sample.parquet")

# Keep only samples with TESSERA embeddings
data = (
    sample_master
    .merge(tessera_embeddings, on="sample_id", how="inner")
    .merge(satclip_embeddings, on="sample_id", how="inner")
)

coord_cols = ["x", "y"]
satclip_cols = [c for c in data.columns if c.startswith("satclip_")]
tessera_cols = [c for c in data.columns if c.startswith("tessera_")]

feature_sets = {
    "Coord_xy": coord_cols,
    "SatCLIP": satclip_cols,
    "TESSERA": tessera_cols,
    "SatCLIP_TESSERA": satclip_cols + tessera_cols
}

print("Evaluation sample:")
print(data["task"].value_counts())
print("Total:", len(data))


def reg_metrics(y_true, pred):
    return {
        "rmse": np.sqrt(mean_squared_error(y_true, pred)),
        "mae": mean_absolute_error(y_true, pred),
        "r2": r2_score(y_true, pred)
    }


def cls_metrics(y_true, pred):
    return {
        "accuracy": accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro")
    }


def evaluate_regression(X, y, groups):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    # Random split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42
    )
    model.fit(X_train, y_train)
    random_result = reg_metrics(y_test, model.predict(X_test))

    # Borough GroupKFold
    fold_results = []
    for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups):
        model.fit(X[train_idx], y[train_idx])
        fold_results.append(reg_metrics(y[test_idx], model.predict(X[test_idx])))

    group_result = pd.DataFrame(fold_results).mean().to_dict()

    return random_result, group_result


def evaluate_classification(X, y, groups):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("ridge_classifier", RidgeClassifier(class_weight="balanced"))
    ])

    # Random split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    model.fit(X_train, y_train)
    random_result = cls_metrics(y_test, model.predict(X_test))

    # Borough GroupKFold
    fold_results = []
    for train_idx, test_idx in GroupKFold(n_splits=5).split(X, y, groups):
        model.fit(X[train_idx], y[train_idx])
        fold_results.append(cls_metrics(y[test_idx], model.predict(X[test_idx])))

    group_result = pd.DataFrame(fold_results).mean().to_dict()

    return random_result, group_result


rows = []

# 1. PTAL regression
ptal = data[data["task"] == "PTAL"].dropna(subset=["label_regression", "borough"]).copy()
y = ptal["label_regression"].to_numpy(dtype=np.float32)
groups = ptal["borough"].to_numpy()

for name, cols in feature_sets.items():
    X = ptal[cols].to_numpy(dtype=np.float32)
    random_result, group_result = evaluate_regression(X, y, groups)

    rows.append({"task": "PTAL", "target_type": "regression", "feature_set": name,
                 "validation": "random_70_30", **random_result})
    rows.append({"task": "PTAL", "target_type": "regression", "feature_set": name,
                 "validation": "borough_groupkfold_5", **group_result})


# 2. PTAL classification
ptal_cls = data[data["task"] == "PTAL"].dropna(subset=["ptal_level", "borough"]).copy()
y = ptal_cls["ptal_level"].astype(str).to_numpy()
groups = ptal_cls["borough"].to_numpy()

for name, cols in feature_sets.items():
    X = ptal_cls[cols].to_numpy(dtype=np.float32)
    random_result, group_result = evaluate_classification(X, y, groups)

    rows.append({"task": "PTAL", "target_type": "classification", "feature_set": name,
                 "validation": "random_70_30", **random_result})
    rows.append({"task": "PTAL", "target_type": "classification", "feature_set": name,
                 "validation": "borough_groupkfold_5", **group_result})


# 3. EPC regression
epc = data[data["task"] == "EPC"].dropna(subset=["label_regression", "borough"]).copy()
y = epc["label_regression"].to_numpy(dtype=np.float32)
groups = epc["borough"].to_numpy()

for name, cols in feature_sets.items():
    X = epc[cols].to_numpy(dtype=np.float32)
    random_result, group_result = evaluate_regression(X, y, groups)

    rows.append({"task": "EPC", "target_type": "regression", "feature_set": name,
                 "validation": "random_70_30", **random_result})
    rows.append({"task": "EPC", "target_type": "regression", "feature_set": name,
                 "validation": "borough_groupkfold_5", **group_result})


tessera_results = pd.DataFrame(rows)
tessera_results.to_csv(TABLE_DIR / "tessera_comparison_results.csv", index=False)

print("Saved:", TABLE_DIR / "tessera_comparison_results.csv")

In [ ]:
# ============================================================
# key TESSERA comparison results
# ============================================================

res = tessera_results.copy()

display(
    res[
        (res["task"] == "PTAL") &
        (res["target_type"] == "regression")
    ][["feature_set", "validation", "rmse", "mae", "r2"]]
    .sort_values(["validation", "rmse"])
)

display(
    res[
        (res["task"] == "PTAL") &
        (res["target_type"] == "classification")
    ][["feature_set", "validation", "accuracy", "macro_f1"]]
    .sort_values(["validation", "macro_f1"], ascending=[True, False])
)

display(
    res[
        (res["task"] == "EPC") &
        (res["target_type"] == "regression")
    ][["feature_set", "validation", "rmse", "mae", "r2"]]
    .sort_values(["validation", "rmse"])
)

**TESSERA result**  
This is the strongest result so far. TESSERA clearly improves over coordinate and SatCLIP baselines, especially for EPC. This suggests EO-visible built-environment and land-cover information is useful for both accessibility and housing-related prediction.  


## Preparing aerial image tile downloads

The DINOv3 experiment requires high-resolution aerial image chips. The raw Digimap aerial imagery is provided as 25cm resolution image tiles, typically organised as 1km × 1km British National Grid tiles. Instead of downloading imagery blindly for the whole of Greater London, this step estimates which 1km tiles are needed to cover the London-wide sample used for DINOv3.

The output is a list of required tile coordinates and larger download batches. This helps keep the aerial image download tied to the actual model sample.

In [ ]:
# ============================================================
# Estimate required 1km aerial tiles for DINOv3 sample crops
# ============================================================

from pathlib import Path

BASE = Path("/content/drive/MyDrive/GEOG0105")
TABLE_DIR = BASE / "Outputs" / "tables"

crop_index_path = TABLE_DIR / "aerial_crop_index_dino_londonwide_sample.csv"

crop_index = pd.read_csv(crop_index_path)

print("Crop index loaded:", crop_index.shape)
print(crop_index["task"].value_counts())
display(crop_index.head())

# Each Digimap aerial source tile is 1km × 1km in British National Grid.
TILE_SIZE = 1000

def tile_range_for_crop(row):
    """
    Return all 1km tile origins intersecting a crop bounding box.
    Tile origin means lower-left easting/northing rounded down to 1000m.
    """
    xmin, xmax = row["xmin"], row["xmax"]
    ymin, ymax = row["ymin"], row["ymax"]

    x_tiles = np.arange(
        np.floor(xmin / TILE_SIZE) * TILE_SIZE,
        np.floor(xmax / TILE_SIZE) * TILE_SIZE + TILE_SIZE,
        TILE_SIZE
    ).astype(int)

    y_tiles = np.arange(
        np.floor(ymin / TILE_SIZE) * TILE_SIZE,
        np.floor(ymax / TILE_SIZE) * TILE_SIZE + TILE_SIZE,
        TILE_SIZE
    ).astype(int)

    return [(x, y) for x in x_tiles for y in y_tiles]

tile_records = []

for _, row in crop_index.iterrows():
    for x0, y0 in tile_range_for_crop(row):
        tile_records.append({
            "tile_x0": x0,
            "tile_y0": y0,
            "sample_id": row["sample_id"],
            "task": row["task"]
        })

tiles_long = pd.DataFrame(tile_records)

required_tiles = (
    tiles_long
    .groupby(["tile_x0", "tile_y0"])
    .agg(
        n_samples=("sample_id", "nunique"),
        tasks=("task", lambda x: ",".join(sorted(set(x))))
    )
    .reset_index()
    .sort_values(["tile_x0", "tile_y0"])
)

required_tiles["tile_name_hint"] = (
    required_tiles["tile_x0"].astype(str) + "_" + required_tiles["tile_y0"].astype(str)
)

print("Required unique 1km tiles:", len(required_tiles))
display(required_tiles.head())
display(required_tiles.tail())

required_tiles.to_csv(TABLE_DIR / "required_aerial_1km_tiles_for_dino_sample.csv", index=False)
print("Saved:", TABLE_DIR / "required_aerial_1km_tiles_for_dino_sample.csv")

In [ ]:
# ============================================================
# Group required 1km tiles into 10km download batches
# ============================================================

BATCH_SIZE = 10000  # 10km × 10km = up to 100 1km tiles

required_tiles["batch_x0"] = (required_tiles["tile_x0"] // BATCH_SIZE) * BATCH_SIZE
required_tiles["batch_y0"] = (required_tiles["tile_y0"] // BATCH_SIZE) * BATCH_SIZE

download_batches = (
    required_tiles
    .groupby(["batch_x0", "batch_y0"])
    .agg(
        n_required_tiles=("tile_x0", "size"),
        n_linked_samples=("n_samples", "sum"),
        x_min=("tile_x0", "min"),
        x_max=("tile_x0", "max"),
        y_min=("tile_y0", "min"),
        y_max=("tile_y0", "max")
    )
    .reset_index()
)

# Download area should be batch boundary, not only min/max tile inside it.
download_batches["download_xmin"] = download_batches["batch_x0"]
download_batches["download_xmax"] = download_batches["batch_x0"] + BATCH_SIZE
download_batches["download_ymin"] = download_batches["batch_y0"]
download_batches["download_ymax"] = download_batches["batch_y0"] + BATCH_SIZE

download_batches = download_batches.sort_values(["batch_x0", "batch_y0"])

print("Number of 10km download batches:", len(download_batches))
display(download_batches)

download_batches.to_csv(TABLE_DIR / "aerial_download_batches_10km.csv", index=False)
print("Saved:", TABLE_DIR / "aerial_download_batches_10km.csv")

In [ ]:
# ============================================================
# Visual check of required aerial tile coverage
# ============================================================

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 8))

plt.scatter(
    required_tiles["tile_x0"],
    required_tiles["tile_y0"],
    s=4,
    label="Required 1km tiles"
)

for _, row in download_batches.iterrows():
    xs = [
        row["download_xmin"], row["download_xmax"], row["download_xmax"],
        row["download_xmin"], row["download_xmin"]
    ]
    ys = [
        row["download_ymin"], row["download_ymin"], row["download_ymax"],
        row["download_ymax"], row["download_ymin"]
    ]
    plt.plot(xs, ys, linewidth=1)

plt.xlabel("British National Grid Easting")
plt.ylabel("British National Grid Northing")
plt.title("Estimated aerial imagery tiles needed for DINOv3 sample")
plt.axis("equal")
plt.legend()
plt.show()

# Post-meeting (June 5th) continuation

So far, the notebook has already built the PTAL and EPC downstream tasks, tested coordinate and SatCLIP location baselines, extracted TESSERA satellite embeddings, and prepared the aerial image download plan.

The next steps are:

1. check which existing outputs are available;
2. complete the DINOv3/v2 aerial image embedding pipeline;
3. add Google AlphaEarth / Satellite Embedding as another satellite representation;
4. add simple structured controls;
5. test whether embeddings can predict the residuals beyond these controls.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import pandas as pd

print("MyDrive exists:", Path("/content/drive/MyDrive").exists())
print("Top-level folders in MyDrive:")
for p in list(Path("/content/drive/MyDrive").iterdir())[:30]:
    print(p)

In [ ]:
# ============================================================
# check existing outputs
# ============================================================

from pathlib import Path
import pandas as pd
import os

BASE = Path("/content/drive/MyDrive/GEOG0105")
OUT = BASE / "Outputs"
TABLE_DIR = OUT / "tables"
EMB_DIR = OUT / "embeddings"

files_to_check = {
    "Clean master sample table": TABLE_DIR / "sample_master_clean.csv",
    "Clean PTAL samples": TABLE_DIR / "ptal_samples_clean.csv",
    "Clean EPC samples": TABLE_DIR / "epc_samples_clean.csv",
    "DINO London-wide sample": TABLE_DIR / "dino_londonwide_sample.csv",
    "DINO aerial crop index": TABLE_DIR / "aerial_crop_index_dino_londonwide_sample.csv",
    "Coordinate baseline results": TABLE_DIR / "coordinate_baseline_results.csv",
    "SatCLIP embeddings": EMB_DIR / "satclip_embeddings_full.parquet",
    "TESSERA embeddings": EMB_DIR / "tessera_embeddings_dino_sample.parquet",
    "Aerial download batches": TABLE_DIR / "aerial_download_batches_compact.csv",
}

status = []
for name, path in files_to_check.items():
    status.append({
        "item": name,
        "exists": path.exists(),
        "path": str(path),
        "size_MB": round(path.stat().st_size / 1024**2, 2) if path.exists() else None
    })

status_df = pd.DataFrame(status)
display(status_df)

In [ ]:
# ============================================================
# Reload core sample tables
# ============================================================

sample_master = pd.read_csv(TABLE_DIR / "sample_master_clean.csv")
ptal_clean = pd.read_csv(TABLE_DIR / "ptal_samples_clean.csv")
epc_clean = pd.read_csv(TABLE_DIR / "epc_samples_clean.csv")
dino_samples = pd.read_csv(TABLE_DIR / "dino_londonwide_sample.csv")
dino_crop_index = pd.read_csv(TABLE_DIR / "aerial_crop_index_dino_londonwide_sample.csv")

print("Clean master samples:", sample_master.shape)
print("PTAL clean samples:", ptal_clean.shape)
print("EPC clean samples:", epc_clean.shape)
print("DINO sample:", dino_samples.shape)
print("DINO crop index:", dino_crop_index.shape)

display(sample_master["task"].value_counts())
display(dino_samples["task"].value_counts())

In [ ]:
# ============================================================
# Reload existing embeddings
# ============================================================

satclip_path = EMB_DIR / "satclip_embeddings_full.parquet"
tessera_path = EMB_DIR / "tessera_embeddings_dino_sample.parquet"

satclip_embeddings = pd.read_parquet(satclip_path) if satclip_path.exists() else None
tessera_embeddings = pd.read_parquet(tessera_path) if tessera_path.exists() else None

if satclip_embeddings is not None:
    print("SatCLIP embeddings:", satclip_embeddings.shape)
else:
    print("SatCLIP embeddings not found.")

if tessera_embeddings is not None:
    print("TESSERA embeddings:", tessera_embeddings.shape)
else:
    print("TESSERA embeddings not found.")

## 2. Test Digimap aerial image batch

This section checks whether the first downloaded Digimap aerial batch can be read correctly, matched with the DINO sample crop index, and used to crop small aerial image chips.

In [ ]:
# ============================================================
# 2.1 Check downloaded Digimap batch files
# ============================================================

from pathlib import Path
import pandas as pd
import os

BASE = Path("/content/drive/MyDrive/GEOG0105")
BATCH_NAME = "batch_500000_170000"

DIGIMAP_DIR = BASE / "Raw Data" / "Digimap" / BATCH_NAME

print("Digimap batch folder:")
print(DIGIMAP_DIR)
print("Exists:", DIGIMAP_DIR.exists())

files = list(DIGIMAP_DIR.rglob("*"))

print("Total files:", len(files))

file_summary = pd.DataFrame({
    "path": [str(p) for p in files],
    "name": [p.name for p in files],
    "suffix": [p.suffix.lower() for p in files],
    "size_MB": [round(p.stat().st_size / 1024**2, 2) if p.is_file() else None for p in files],
})

display(file_summary["suffix"].value_counts())
display(file_summary.head(30))

In [ ]:
# ============================================================
# 2.2 Install and import rasterio
# ============================================================

try:
    import rasterio
    print("rasterio already installed:", rasterio.__version__)
except ImportError:
    !pip -q install rasterio
    import rasterio
    print("rasterio installed:", rasterio.__version__)

import numpy as np
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds

In [ ]:
# ============================================================
# 2.3 Test reading one Digimap image with georeferencing
# ============================================================

jpg_files = sorted(list(DIGIMAP_DIR.rglob("*.jpg")) + list(DIGIMAP_DIR.rglob("*.jpeg")))

print("Number of JPG files:", len(jpg_files))

test_img = jpg_files[0]
print("Test image:", test_img.name)

with rasterio.open(test_img) as src:
    print("CRS:", src.crs)
    print("Width, height:", src.width, src.height)
    print("Count bands:", src.count)
    print("Transform:", src.transform)
    print("Bounds:", src.bounds)

In [ ]:
# ============================================================
# 2.4 Build raster index for this batch
# ============================================================

raster_records = []

for img_path in jpg_files:
    try:
        with rasterio.open(img_path) as src:
            b = src.bounds
            raster_records.append({
                "path": str(img_path),
                "name": img_path.name,
                "left": b.left,
                "bottom": b.bottom,
                "right": b.right,
                "top": b.top,
                "width": src.width,
                "height": src.height,
                "bands": src.count,
                "crs": str(src.crs),
            })
    except Exception as e:
        print("Failed:", img_path.name, e)

raster_index = pd.DataFrame(raster_records)

print("Raster files indexed:", len(raster_index))
display(raster_index.head())

print("Batch raster bounds:")
print("xmin:", raster_index["left"].min())
print("ymin:", raster_index["bottom"].min())
print("xmax:", raster_index["right"].max())
print("ymax:", raster_index["top"].max())

In [ ]:
# ============================================================
# 2.5 Load DINO crop index and identify samples inside this batch
# ============================================================

TABLE_DIR = BASE / "Outputs" / "tables"

crop_index_path = TABLE_DIR / "aerial_crop_index_dino_londonwide_sample.csv"
crop_index = pd.read_csv(crop_index_path)

print("Crop index shape:", crop_index.shape)
print("Columns:")
print(crop_index.columns.tolist())

display(crop_index.head())

In [ ]:
# ============================================================
# Auto-detect crop bbox columns
# ============================================================

cols = crop_index.columns.tolist()

def find_first(possible_names):
    for name in possible_names:
        if name in cols:
            return name
    return None

xmin_col = find_first(["xmin", "crop_xmin", "bbox_xmin", "left", "x_min"])
ymin_col = find_first(["ymin", "crop_ymin", "bbox_ymin", "bottom", "y_min"])
xmax_col = find_first(["xmax", "crop_xmax", "bbox_xmax", "right", "x_max"])
ymax_col = find_first(["ymax", "crop_ymax", "bbox_ymax", "top", "y_max"])

print("Detected bbox columns:")
print(xmin_col, ymin_col, xmax_col, ymax_col)

if None in [xmin_col, ymin_col, xmax_col, ymax_col]:
    raise ValueError("Could not automatically detect crop bbox columns. Please show the crop_index columns.")

In [ ]:
# ============================================================
# Find crop samples covered by the current batch extent
# ============================================================

batch_xmin = raster_index["left"].min()
batch_ymin = raster_index["bottom"].min()
batch_xmax = raster_index["right"].max()
batch_ymax = raster_index["top"].max()

samples_in_batch = crop_index[
    (crop_index[xmin_col] >= batch_xmin) &
    (crop_index[xmax_col] <= batch_xmax) &
    (crop_index[ymin_col] >= batch_ymin) &
    (crop_index[ymax_col] <= batch_ymax)
].copy()

print("Samples fully inside this batch:", len(samples_in_batch))

if "task" in samples_in_batch.columns:
    display(samples_in_batch["task"].value_counts())

display(samples_in_batch.head())

In [ ]:
# ============================================================
# 2.6 Find samples fully covered by a single image tile
# ============================================================

single_tile_matches = []

for _, tile in raster_index.iterrows():
    subset = samples_in_batch[
        (samples_in_batch[xmin_col] >= tile["left"]) &
        (samples_in_batch[xmax_col] <= tile["right"]) &
        (samples_in_batch[ymin_col] >= tile["bottom"]) &
        (samples_in_batch[ymax_col] <= tile["top"])
    ].copy()

    if len(subset) > 0:
        subset["tile_path"] = tile["path"]
        subset["tile_name"] = tile["name"]
        single_tile_matches.append(subset)

if single_tile_matches:
    single_tile_samples = pd.concat(single_tile_matches, ignore_index=True)
else:
    single_tile_samples = pd.DataFrame()

print("Samples fully inside a single tile:", len(single_tile_samples))

if len(single_tile_samples) > 0:
    if "task" in single_tile_samples.columns:
        display(single_tile_samples["task"].value_counts())
    display(single_tile_samples.head())
else:
    print("No single-tile samples found. We may need to handle mosaic / cross-tile crops.")

In [ ]:
# ============================================================
# 2.7 Crop and display a few test aerial chips
# ============================================================

def crop_one_sample(row):
    img_path = row["tile_path"]
    xmin = row[xmin_col]
    ymin = row[ymin_col]
    xmax = row[xmax_col]
    ymax = row[ymax_col]

    with rasterio.open(img_path) as src:
        window = from_bounds(xmin, ymin, xmax, ymax, transform=src.transform)
        img = src.read(window=window)

    # rasterio reads as (bands, height, width); convert to (height, width, bands)
    img = np.moveaxis(img, 0, -1)

    # keep RGB if there are more than 3 bands
    if img.shape[-1] > 3:
        img = img[:, :, :3]

    return img

n_show = min(6, len(single_tile_samples))
test_samples = single_tile_samples.sample(n_show, random_state=42) if n_show > 0 else pd.DataFrame()

for i, (_, row) in enumerate(test_samples.iterrows()):
    img = crop_one_sample(row)

    print("\nSample:", row.get("sample_id", "unknown"))
    print("Task:", row.get("task", "unknown"))
    print("Tile:", row["tile_name"])
    print("Chip shape:", img.shape)

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{row.get('task', '')} | {row.get('sample_id', '')}")
    plt.show()

In [ ]:
# ============================================================
# 2.8 Check all downloaded Digimap folders
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/GEOG0105")
TABLE_DIR = BASE / "Outputs" / "tables"
DIGIMAP_ROOT = BASE / "Raw Data" / "Digimap"

print("Digimap root:", DIGIMAP_ROOT)
print("Exists:", DIGIMAP_ROOT.exists())

batch_folders = sorted([
    p for p in DIGIMAP_ROOT.iterdir()
    if p.is_dir() and p.name.startswith("batch_")
])

print("Batch folders found:", len(batch_folders))

summary = []

for folder in batch_folders:
    jpgs = list(folder.rglob("*.jpg")) + list(folder.rglob("*.jpeg"))
    jgws = list(folder.rglob("*.jgw"))
    jpws = list(folder.rglob("*.jpw"))
    wlds = list(folder.rglob("*.wld"))
    xmls = list(folder.rglob("*.xml"))
    txts = list(folder.rglob("*.txt"))

    size_gb = sum(
        p.stat().st_size for p in folder.rglob("*") if p.is_file()
    ) / 1024**3

    summary.append({
        "batch": folder.name,
        "jpg_count": len(jpgs),
        "jgw_count": len(jgws),
        "jpw_count": len(jpws),
        "wld_count": len(wlds),
        "xml_count": len(xmls),
        "txt_count": len(txts),
        "size_GB": round(size_gb, 2)
    })

download_check = pd.DataFrame(summary)

display(download_check)

print("Total batches:", len(download_check))
print("Total JPG files:", download_check["jpg_count"].sum())
print("Total size GB:", round(download_check["size_GB"].sum(), 2))

# 检查有没有空文件夹
display(download_check[download_check["jpg_count"] == 0])

In [ ]:
# ============================================================
# 2.9 Build raster index for all Digimap aerial tiles
# 读取所有 jpg 的 bounds，建立完整 tile index
# ============================================================

try:
    import rasterio
    print("rasterio already installed:", rasterio.__version__)
except ImportError:
    !pip -q install rasterio
    import rasterio
    print("rasterio installed:", rasterio.__version__)

from tqdm.auto import tqdm

all_jpgs = []

for folder in batch_folders:
    all_jpgs.extend(list(folder.rglob("*.jpg")) + list(folder.rglob("*.jpeg")))

print("Total JPGs found:", len(all_jpgs))

records = []

for img_path in tqdm(all_jpgs):
    try:
        with rasterio.open(img_path) as src:
            b = src.bounds

            batch_name = None
            for part in img_path.parts:
                if part.startswith("batch_"):
                    batch_name = part
                    break

            records.append({
                "batch": batch_name,
                "path": str(img_path),
                "name": img_path.name,
                "left": b.left,
                "bottom": b.bottom,
                "right": b.right,
                "top": b.top,
                "width": src.width,
                "height": src.height,
                "bands": src.count,
                "crs": str(src.crs),
                "pixel_width": src.transform.a,
                "pixel_height": abs(src.transform.e),
                "error": None
            })
    except Exception as e:
        records.append({
            "batch": None,
            "path": str(img_path),
            "name": img_path.name,
            "left": None,
            "bottom": None,
            "right": None,
            "top": None,
            "width": None,
            "height": None,
            "bands": None,
            "crs": None,
            "pixel_width": None,
            "pixel_height": None,
            "error": str(e)
        })

raster_index_all = pd.DataFrame(records)

print("Raster index shape:", raster_index_all.shape)
print("Errors:", raster_index_all["error"].notna().sum())

display(raster_index_all.head())

print("Overall downloaded bounds:")
print("xmin:", raster_index_all["left"].min())
print("ymin:", raster_index_all["bottom"].min())
print("xmax:", raster_index_all["right"].max())
print("ymax:", raster_index_all["top"].max())

display(raster_index_all[["width", "height", "pixel_width", "pixel_height"]].describe())

raster_index_path = TABLE_DIR / "digimap_raster_index_all.csv"
raster_index_all.to_csv(raster_index_path, index=False)

print("Saved raster index to:", raster_index_path)

In [ ]:
# ============================================================
# 2.10 Match DINO samples to aerial tiles
# 找到哪些 DINO samples 可以直接从单张 1km tile 裁出来
# ============================================================

crop_index_path = TABLE_DIR / "aerial_crop_index_dino_londonwide_sample.csv"
crop_index = pd.read_csv(crop_index_path)

print("Crop index shape:", crop_index.shape)
print("Crop index columns:")
print(crop_index.columns.tolist())

cols = crop_index.columns.tolist()

def find_first(possible_names):
    for name in possible_names:
        if name in cols:
            return name
    return None

xmin_col = find_first(["xmin", "crop_xmin", "bbox_xmin", "left", "x_min", "crop_left"])
ymin_col = find_first(["ymin", "crop_ymin", "bbox_ymin", "bottom", "y_min", "crop_bottom"])
xmax_col = find_first(["xmax", "crop_xmax", "bbox_xmax", "right", "x_max", "crop_right"])
ymax_col = find_first(["ymax", "crop_ymax", "bbox_ymax", "top", "y_max", "crop_top"])

print("Detected bbox columns:", xmin_col, ymin_col, xmax_col, ymax_col)

if None in [xmin_col, ymin_col, xmax_col, ymax_col]:
    raise ValueError("Could not detect crop bbox columns. Check crop_index columns above.")

# 只保留没有读取错误的 tiles
tiles = raster_index_all[raster_index_all["error"].isna()].copy()

single_tile_matches = []

for _, tile in tqdm(tiles.iterrows(), total=len(tiles)):
    subset = crop_index[
        (crop_index[xmin_col] >= tile["left"]) &
        (crop_index[xmax_col] <= tile["right"]) &
        (crop_index[ymin_col] >= tile["bottom"]) &
        (crop_index[ymax_col] <= tile["top"])
    ].copy()

    if len(subset) > 0:
        subset["tile_path"] = tile["path"]
        subset["tile_name"] = tile["name"]
        subset["source_batch"] = tile["batch"]
        single_tile_matches.append(subset)

if single_tile_matches:
    single_tile_samples_all = pd.concat(single_tile_matches, ignore_index=True)
    single_tile_samples_all = single_tile_samples_all.drop_duplicates(subset=["sample_id"])
else:
    single_tile_samples_all = pd.DataFrame()

print("Samples fully inside a single tile:", len(single_tile_samples_all), "/", len(crop_index))

if len(single_tile_samples_all) > 0 and "task" in single_tile_samples_all.columns:
    display(single_tile_samples_all["task"].value_counts())

# 没有完全落在单张 tile 里的样本，一般是 crop 跨了 tile 边界
single_ids = set(single_tile_samples_all["sample_id"]) if len(single_tile_samples_all) > 0 else set()
not_single_tile = crop_index[~crop_index["sample_id"].isin(single_ids)].copy()

print("Not single-tile samples:", len(not_single_tile), "/", len(crop_index))

if "task" in not_single_tile.columns:
    display(not_single_tile["task"].value_counts())

# 保存两个 index
single_tile_index_path = TABLE_DIR / "aerial_single_tile_samples_all.csv"
not_single_tile_path = TABLE_DIR / "aerial_not_single_tile_samples.csv"

single_tile_samples_all.to_csv(single_tile_index_path, index=False)
not_single_tile.to_csv(not_single_tile_path, index=False)

print("Saved single-tile sample index:", single_tile_index_path)
print("Saved not-single-tile sample index:", not_single_tile_path)

display(single_tile_samples_all.head())

In [ ]:
# ============================================================
# 2.11 Preview several full-London crops
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from rasterio.windows import from_bounds

def crop_one_single_tile(row):
    with rasterio.open(row["tile_path"]) as src:
        window = from_bounds(
            row[xmin_col],
            row[ymin_col],
            row[xmax_col],
            row[ymax_col],
            transform=src.transform
        )
        img = src.read(window=window)

    img = np.moveaxis(img, 0, -1)

    if img.shape[-1] > 3:
        img = img[:, :, :3]

    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

preview = single_tile_samples_all.sample(
    min(8, len(single_tile_samples_all)),
    random_state=42
)

for _, row in preview.iterrows():
    img = crop_one_single_tile(row)

    print("Sample:", row.get("sample_id", "unknown"))
    print("Task:", row.get("task", "unknown"))
    print("Batch:", row.get("source_batch", "unknown"))
    print("Tile:", row.get("tile_name", "unknown"))
    print("Chip shape:", img.shape)

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{row.get('task', '')} | {row.get('source_batch', '')}")
    plt.show()

In [ ]:
# ============================================================
# 2.12 Prepare mosaic candidates for cross-tile samples
# 找出 not-single-tile samples 需要哪些相邻 tiles
# ============================================================

from tqdm.auto import tqdm
import pandas as pd
import numpy as np

# Make sure these already exist from previous cells:
# crop_index
# raster_index_all
# single_tile_samples_all
# not_single_tile
# xmin_col, ymin_col, xmax_col, ymax_col

tiles = raster_index_all[raster_index_all["error"].isna()].copy()

mosaic_records = []

for _, row in tqdm(not_single_tile.iterrows(), total=len(not_single_tile)):
    xmin = row[xmin_col]
    xmax = row[xmax_col]
    ymin = row[ymin_col]
    ymax = row[ymax_col]

    # Find all tiles that intersect this crop bbox
    candidate_tiles = tiles[
        (tiles["right"] > xmin) &
        (tiles["left"] < xmax) &
        (tiles["top"] > ymin) &
        (tiles["bottom"] < ymax)
    ].copy()

    mosaic_records.append({
        "sample_id": row["sample_id"],
        "task": row["task"],
        "x": row["x"],
        "y": row["y"],
        "borough": row["borough"],
        "label_regression": row["label_regression"],
        "label_classification": row["label_classification"],
        "crop_size_m": row["crop_size_m"],
        xmin_col: xmin,
        xmax_col: xmax,
        ymin_col: ymin,
        ymax_col: ymax,
        "n_intersecting_tiles": len(candidate_tiles),
        "tile_paths": "|".join(candidate_tiles["path"].tolist()),
        "tile_names": "|".join(candidate_tiles["name"].tolist()),
        "source_batches": "|".join(sorted(candidate_tiles["batch"].dropna().unique().tolist())),
    })

mosaic_samples_all = pd.DataFrame(mosaic_records)

print("Mosaic candidate samples:", len(mosaic_samples_all))
display(mosaic_samples_all["n_intersecting_tiles"].value_counts().sort_index())

if "task" in mosaic_samples_all.columns:
    display(mosaic_samples_all["task"].value_counts())

# Check if any sample has no intersecting tile
missing_mosaic = mosaic_samples_all[mosaic_samples_all["n_intersecting_tiles"] == 0]
print("Samples with no intersecting tiles:", len(missing_mosaic))

display(mosaic_samples_all.head())

mosaic_index_path = TABLE_DIR / "aerial_mosaic_samples_all.csv"
mosaic_samples_all.to_csv(mosaic_index_path, index=False)

print("Saved mosaic sample index:", mosaic_index_path)

In [ ]:
# ============================================================
# 2.13 Test mosaic crop for cross-tile samples
# 测试跨 tile 样本是否可以正常拼接并裁图
# ============================================================

import rasterio
from rasterio.merge import merge
import matplotlib.pyplot as plt
import numpy as np

def crop_one_mosaic(row):
    tile_paths = row["tile_paths"].split("|")
    tile_paths = [p for p in tile_paths if len(p) > 0]

    if len(tile_paths) == 0:
        raise ValueError("No tile paths for this sample.")

    bounds = (
        row[xmin_col],
        row[ymin_col],
        row[xmax_col],
        row[ymax_col],
    )

    srcs = [rasterio.open(p) for p in tile_paths]

    try:
        mosaic, transform = merge(srcs, bounds=bounds, res=(0.25, 0.25))
    finally:
        for src in srcs:
            src.close()

    img = np.moveaxis(mosaic, 0, -1)

    if img.shape[-1] > 3:
        img = img[:, :, :3]

    img = np.clip(img, 0, 255).astype(np.uint8)
    return img

preview_mosaic = mosaic_samples_all[
    mosaic_samples_all["n_intersecting_tiles"] > 1
].sample(
    min(8, (mosaic_samples_all["n_intersecting_tiles"] > 1).sum()),
    random_state=42
)

for _, row in preview_mosaic.iterrows():
    img = crop_one_mosaic(row)

    print("Sample:", row["sample_id"])
    print("Task:", row["task"])
    print("Tiles:", row["n_intersecting_tiles"])
    print("Batches:", row["source_batches"])
    print("Chip shape:", img.shape)

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.axis("off")
    plt.title(f"{row['task']} | {row['n_intersecting_tiles']} tiles")
    plt.show()

In [ ]:
# ============================================================
# 2.14 Build final aerial crop source index
# 合并 single-tile 和 mosaic samples，形成 DINOv3 输入索引
# ============================================================

single_for_final = single_tile_samples_all.copy()
single_for_final["crop_mode"] = "single_tile"
single_for_final["tile_paths"] = single_for_final["tile_path"]
single_for_final["tile_names"] = single_for_final["tile_name"]
single_for_final["source_batches"] = single_for_final["source_batch"]

mosaic_for_final = mosaic_samples_all.copy()
mosaic_for_final["crop_mode"] = "mosaic"

# Keep common columns
common_cols = [
    "sample_id", "task", "x", "y", "borough",
    "label_regression", "label_classification",
    "crop_size_m",
    xmin_col, xmax_col, ymin_col, ymax_col,
    "crop_mode", "tile_paths", "tile_names", "source_batches"
]

aerial_crop_source_index = pd.concat(
    [
        single_for_final[common_cols],
        mosaic_for_final[common_cols]
    ],
    ignore_index=True
)

# Remove duplicates if any
aerial_crop_source_index = aerial_crop_source_index.drop_duplicates(
    subset=["sample_id"],
    keep="first"
)

print("Final aerial crop source index:", aerial_crop_source_index.shape)
display(aerial_crop_source_index["crop_mode"].value_counts())
display(aerial_crop_source_index["task"].value_counts())

missing_ids = set(crop_index["sample_id"]) - set(aerial_crop_source_index["sample_id"])
print("Missing samples after combining:", len(missing_ids))

final_index_path = TABLE_DIR / "aerial_crop_source_index_all.csv"
aerial_crop_source_index.to_csv(final_index_path, index=False)

print("Saved final aerial crop source index:", final_index_path)
display(aerial_crop_source_index.head())